# Demand Forecasting — Execução no Google Colab

> **Antes de iniciar:** `Runtime → Change runtime type → T4 GPU`
>
> Os checkpoints são salvos no Google Drive para que treinos interrompidos possam ser retomados.

In [ ]:
# ── 1. Montar Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# Pasta base no Drive onde o projeto e os checkpoints serão mantidos
DRIVE_PROJECT = '/content/drive/MyDrive/RNA_demand_forecasting'

import os
os.makedirs(DRIVE_PROJECT, exist_ok=True)
print('Drive montado em:', DRIVE_PROJECT)

In [ ]:
# ── 2. Instalar dependências ──────────────────────────────────────────────────
# TensorFlow vem pré-instalado no Colab; reinstalamos para garantir compatibilidade.
%pip install -q \
    pytorch-forecasting==1.7.0 \
    lightning==2.6.1 \
    statsmodels==0.14.6 \
    joblib

# Verifica GPU disponível para PyTorch
import torch
print('CUDA disponível:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

import tensorflow as tf
print('GPUs TF:', tf.config.list_physical_devices('GPU'))

In [ ]:
# ── 3. Clonar / atualizar repositório ────────────────────────────────────────
REPO_URL = 'https://github.com/VanthuirMaia/demand-forecasting.git'  # ajuste se necessário
REPO_DIR = '/content/demand-forecasting'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

%cd {REPO_DIR}
print('Diretório de trabalho:', os.getcwd())

In [ ]:
# ── 4. Fazer upload do dataset (somente na primeira vez) ──────────────────────
# Se o arquivo já estiver no Drive, copia de lá. Caso contrário, faz upload manual.

DATA_DRIVE = os.path.join(DRIVE_PROJECT, 'train.csv')
DATA_LOCAL = 'data/raw/train.csv'

os.makedirs('data/raw', exist_ok=True)

if os.path.exists(DATA_DRIVE):
    import shutil
    shutil.copy(DATA_DRIVE, DATA_LOCAL)
    print('Dataset copiado do Drive.')
elif not os.path.exists(DATA_LOCAL):
    print('Dataset não encontrado. Faça o upload abaixo:')
    from google.colab import files
    uploaded = files.upload()  # selecione train.csv
    fname = list(uploaded.keys())[0]
    shutil.move(fname, DATA_LOCAL)
    shutil.copy(DATA_LOCAL, DATA_DRIVE)  # salva no Drive para próximas sessões
    print('Dataset salvo em', DATA_LOCAL)
else:
    print('Dataset já presente:', DATA_LOCAL)

In [ ]:
# ── 5. Sincronizar checkpoints do Drive (retomada de treino) ──────────────────
import shutil

CKPT_DRIVE = os.path.join(DRIVE_PROJECT, 'checkpoints')
CKPT_LOCAL = 'checkpoints'

if os.path.exists(CKPT_DRIVE):
    shutil.copytree(CKPT_DRIVE, CKPT_LOCAL, dirs_exist_ok=True)
    print('Checkpoints restaurados do Drive.')
else:
    print('Nenhum checkpoint anterior encontrado — iniciando do zero.')

In [ ]:
# ── 6. Executar pipeline completo ─────────────────────────────────────────────
# O main.py detecta GPU automaticamente e pula modelos já treinados (checkpoints).
%run main.py

In [ ]:
# ── 7. Salvar checkpoints e resultados no Drive ───────────────────────────────
shutil.copytree(CKPT_LOCAL, CKPT_DRIVE, dirs_exist_ok=True)
shutil.copytree('outputs', os.path.join(DRIVE_PROJECT, 'outputs'), dirs_exist_ok=True)
print('Checkpoints e resultados salvos no Drive.')

In [ ]:
# ── 8. Visualizar figuras geradas ─────────────────────────────────────────────
from IPython.display import Image, display

for fig in ['outputs/figures/comparativo_metricas.png',
            'outputs/figures/predicoes_comparativo.png']:
    if os.path.exists(fig):
        display(Image(fig))